In [99]:
from utils import *

In [100]:
envs_list = [
    "./envs/known_envs/doorkey-5x5-normal.env",
    "./envs/known_envs/doorkey-6x6-direct.env",
    "./envs/known_envs/doorkey-6x6-normal.env",
    "./envs/known_envs/doorkey-6x6-shortcut.env",
    "./envs/known_envs/doorkey-8x8-direct.env",
    "./envs/known_envs/doorkey-8x8-normal.env",
    "./envs/known_envs/doorkey-8x8-shortcut.env",
    "./envs/known_envs/example-8x8.env"
]

In [101]:
def gen_pf(env_path):
    """Determines p(x'|x, u),also stores cost.
    state = (agent_x, agent_y, agent_dir, has_key, door_status)
    action = (MF, TL, TR, PK, UD)"""

    Pf = {}
    env, info = load_env(env_path)

    H = info['height']
    W = info['width']
    
    MF = 0  # Move Forward
    TL = 1  # Turn Left
    TR = 2  # Turn Right
    PK = 3  # Pickup Key
    UD = 4  # Unlock Door

    # 0=right, 1=down, 2=left, 3=up
    #directions
    dir_dict = {0: (1,0), 1: (0,1), 2: (-1,0), 3: (0,-1)}

    #s = x + W * (y + H * (dir + 4 * (key_state + 2 * door_state)))
    # x = s % W
    # y = (s // W) % H
    # dir = (s // (W * H)) % 4
    # key_state = (s // (W * H * 4)) % 2
    # door_state = (s // (W * H * 4 * 2)) % 2

    #determine key position
    key_x, key_y = info['key_pos']
    door_x, door_y = info['door_pos']
  
    for x in range(W):
        for y in range(H):
            for dir in range(4):
                for key_state in [0,1]: #binary has key or doesnt TRUE or FALSE
                    for door_state in [0, 1]:
                        for action in [MF, TL, TR, PK, UD]:
                            # check if placed in wall or closed door 
                            if isinstance(env.grid.get(x, y), Wall) or (isinstance(env.grid.get(x, y), Door) and door_state == 0):
                                continue
                            #check if placed in goal state, dont include in Pf, will set those up in DP seperately
                            if x == info['goal_pos'][0] and y == info['goal_pos'][1]:
                                continue

                            if action == MF:
                                #just a move forward, key and door shouldnt change
                                dx, dy = dir_dict[dir]
                                next_x, next_y = x + dx, y + dy
                                env.grid.get(next_x, next_y)
                                if isinstance(env.grid.get(next_x, next_y), Wall) or (isinstance(env.grid.get(next_x, next_y), Door) and door_state == 0):
                                    next_x, next_y = x, y
                                next_dir = dir
                                next_key = key_state
                                next_door = door_state
                            if action == TL:
                                next_dir = (dir - 1) % 4
                                next_x, next_y = x, y
                                next_key = key_state
                                next_door = door_state
                            if action == TR:
                                next_dir = (dir + 1) % 4
                                next_x, next_y = x, y
                                next_key = key_state
                                next_door = door_state
                            if action == PK:
                                dx, dy = dir_dict[dir]
                                next_dir = dir
                                next_x, next_y = x, y
                                front_x, front_y = x + dx, y + dy
                                #can only pick up key if on the key cell and not already carrying a key
                                if (front_x,front_y) == (key_x, key_y) and key_state == 0:
                                    next_key = 1
                                else:
                                    next_key = key_state
                                next_door = door_state
                            if action == UD:
                                dx, dy = dir_dict[dir]
                                next_dir = dir
                                front_x, front_y = x + dx, y + dy
                                next_x, next_y = x, y
                                next_key = key_state
                                #can only unlock door if on the door cell and carrying a key and door is currently locked
                                if (front_x, front_y) == (door_x, door_y) and key_state == 1 and door_state == 0:
                                    next_door = 1
                                else:
                                    next_door = door_state
                            
                            

                            #compute state scalar 
                            state = x + W * (y + H * (dir + 4 * (key_state + 2 * door_state)))
                            if state not in Pf:
                                Pf[state] = {}
                            
                            next_state = next_x + W * (next_y + H * (next_dir + 4 * (int(next_key) + 2 * int(next_door))))
                            

                            #can index ouut of this using the state value 
                            cost = 1
                            Pf[state][action] = (next_state, cost)

    return Pf

In [102]:
def dynamic_programming(env_path, Pf, gamma):
    """"
    Dynamic Programming Implementation:
    Inputs: env, gamma, actions
    Uses env to extrapoated necessary information about the MDP
    We assume the envirnoment is not partially observable, and we learn the pf from the env directly.
    Returns: V, policy """
    #load env, extract dictionary to built state space 
    
    env, info = load_env(env_path)
    H = info['height']
    W = info['width']
    goal = info['goal_pos']
    #determine key position
    key_x, key_y = info['key_pos']
    door_x, door_y = info['door_pos']
    #time horizon 
    T = W * H 
    
    MF = 0  # Move Forward
    TL = 1  # Turn Left
    TR = 2  # Turn Right
    PK = 3  # Pickup Key
    UD = 4  # Unlock Door

    # Pf = gen_pf(env_path) #does not include goal states
    #pf has form pf[state][action] = (next_state, cost) the transitions are deterministic 
    num_states = W*H*4*2*2 #total states not just walkable  
    num_actions = 5

    
    q = 0 #terminal cost
    
    #acts as V_T for the final time step 
    V = np.full(num_states, np.inf) #populate initial V with inf, since we want to minimuze cost
    #set V of goal states to 0
    #caluclate goal states 
    goal_states = []
    for x in range(W):
        for y in range(H):
            for dir in range(4):
                for key_state in [0,1]: #binary has key or doesnt TRUE or FALSE
                    for door_state in [0, 1]:
                        state = x + W * (y + H * (dir + 4 * (key_state + 2 * door_state)))
                        if x == goal[0] and y == goal[1]:
                            V[state] = q
                            goal_states.append(state)



    pi = np.zeros((T, num_states), dtype=int) #initialize policy array
    for t in range(T-1 , -1, -1): #loop from T-1 to 0
        q_table = np.full((num_states, num_actions), np.inf)
        for x in range(num_states):
            if x not in Pf:
                continue
            if x in goal_states:
                continue
            for u in range(num_actions):
                q_table[x, u] = Pf[x][u][1] + gamma * V[Pf[x][u][0]] #prob of next state #cost(from pf) + gamma * V(next state)
                #not all states in V have a Pf counterpart, keep it at inf
            
            V_t = np.min(q_table[x, :]) #find the min cost action for each state

            #once action row has been computed for each state, update V and pi
            V[x] = V_t #update V for the current state
                #extract policy
            pi[t, x] = np.argmin(q_table[x, :]) #find the action that minimizes cost for each state

    return V, pi


In [103]:
def test_policy(pi, Pf, env_path):

    #s = x + W * (y + H * (dir + 4 * (key_state + 2 * door_state)))
    env, info = load_env(env_path)
    
    H = info['height']
    W = info['width']
    x, y = env.agent_pos 
    dir = env.agent_dir
    key_state = int(env.carrying is not None)
    door = env.grid.get(info['door_pos'][0], info['door_pos'][1])
    door_state = int(not door.is_locked)

    state = x + W * (y + H * (dir + 4 * (key_state + 2 * door_state)))

    sequence = []

    goal_x, goal_y = info['goal_pos']
    for t in range(len(pi)):
        if x == goal_x and y == goal_y:
            break
        action = pi[t, state]
        sequence.append(action)
        state = Pf[state][action][0]


        x = state % W
        y = (state // W) % H
    return sequence


In [104]:
#process all  
seq_list = []
for i in range(len(envs_list)):
    
    env_path = envs_list[i]
    env_folder = f"{env_path}.env"
    env, info = load_env(env_path)
    Pf = gen_pf(env_path)
    print(env_path)
    v, pi = dynamic_programming(env_path, Pf, gamma = 1)
    seq = test_policy(pi, Pf, env_path)
    seq_list.append(seq)
    env_name = os.path.splitext(os.path.basename(env_path))[0]
    draw_gif_from_seq(seq, env, path=f"./gif/{env_name}.gif")


./envs/known_envs/doorkey-5x5-normal.env
GIF is written to ./gif/doorkey-5x5-normal.gif
./envs/known_envs/doorkey-6x6-direct.env
GIF is written to ./gif/doorkey-6x6-direct.gif
./envs/known_envs/doorkey-6x6-normal.env
GIF is written to ./gif/doorkey-6x6-normal.gif
./envs/known_envs/doorkey-6x6-shortcut.env
GIF is written to ./gif/doorkey-6x6-shortcut.gif
./envs/known_envs/doorkey-8x8-direct.env
GIF is written to ./gif/doorkey-8x8-direct.gif
./envs/known_envs/doorkey-8x8-normal.env
GIF is written to ./gif/doorkey-8x8-normal.gif
./envs/known_envs/doorkey-8x8-shortcut.env
GIF is written to ./gif/doorkey-8x8-shortcut.gif
./envs/known_envs/example-8x8.env
GIF is written to ./gif/example-8x8.gif


In [105]:
#make individual gifs
env, info = load_env(envs_list[1])
seq = seq_list[1]
draw_gif_from_seq(seq, env)

GIF is written to ./gif/doorkey.gif


for part B, need to modify the DP alg function and the gen_PF function to accommodate more states

In [ ]:
# expanded state df now includes Denote the index of the top-left cell as (0, 0), bottom-left as (0, 9), top-right

#only thing that is random is key and goal position.
# keys are at (2, 2), (2, 3), (1, 6). Goal is at (6, 1), (7, 3), (6, 6)
# doors are at (5, 3) and (5, 7). Each door can either be open or locked 
# walls are at (5,0), (5,1), (5,2), (5,4), (5,5), (5,6), (5,8), (5,9)
#start always at (4, 8) facing up

In [106]:
def gen_pf_rand():
    """Determines p(x'|x, u),also stores cost.
    state = (agent_x, agent_y, agent_dir, has_key, door_status, key_pos, goal_pos)
    action = (MF, TL, TR, PK, UD)
    walls and doors are fixed"""

    Pf = {}
    #fixed in place 
    Walls = [(5,0), (5,1), (5,2), (5,4), (5,5), (5,6), (5,8), (5,9)]
    Doors = [(5, 3), (5, 7)]
    
    #H, W hard coded to 10 
    H = 10
    W = 10
    
    MF = 0  # Move Forward
    TL = 1  # Turn Left
    TR = 2  # Turn Right
    PK = 3  # Pickup Key
    UD = 4  # Unlock Door

    # 0=right, 1=down, 2=left, 3=up
    #directions
    dir_dict = {0: (1,0), 1: (0,1), 2: (-1,0), 3: (0,-1)}


    #goals
    Goals = [(6, 1), (7, 3), (6, 6)]
    Keys = [(2, 2), (2, 3), (1, 6)]

    for x in range(1,9): # range(9) would include 0, which is not walkable
        for y in range(1,9):
            for dir in range(4):
                for key_state in [0,1]: #binary has key or doesnt TRUE or FALSE
                    for door1_state in [0, 1]:
                        for door2_state in [0, 1]:
                            for key_pos in [(2, 2), (2, 3), (1, 6)]:
                                for goal_pos in [(6, 1), (7, 3), (6, 6)]:
                                    for action in [MF, TL, TR, PK, UD]:
                                        # check if placed in wall or closed door 
                                        if (x, y) in Walls:
                                            continue
                                        if (x, y) == (5, 7) and door2_state == 0:
                                            continue
                                        if (x, y) == (5, 3) and door1_state == 0:
                                            continue
                                        #check if placed in goal state, dont include in Pf, will set those up in DP seperately
                                        if (x, y) == goal_pos:
                                            continue
                                   
                                        if action == MF:
                                            #just a move forward, key and door shouldnt change
                                            dx, dy = dir_dict[dir]
                                            next_x, next_y = x + dx, y + dy
                                            if next_x < 0 or next_x >= 10 or next_y < 0 or next_y >= 10:
                                                next_x, next_y = x, y
                                            if (next_x, next_y) in Walls or ((next_x, next_y) == (5, 3) and door1_state == 0) or ((next_x, next_y) == (5, 7) and door2_state == 0):
                                                next_x, next_y = x, y
                                            next_dir = dir
                                            next_key = key_state
                                            next_door1 = door1_state
                                            next_door2 = door2_state
                                        if action == TL:
                                            next_dir = (dir - 1) % 4
                                            next_x, next_y = x, y
                                            next_key = key_state
                                            next_door1 = door1_state
                                            next_door2 = door2_state
                                        if action == TR:
                                            next_dir = (dir + 1) % 4
                                            next_x, next_y = x, y
                                            next_key = key_state
                                            next_door1 = door1_state
                                            next_door2 = door2_state
                                        if action == PK:
                                            dx, dy = dir_dict[dir]
                                            next_dir = dir
                                            next_x, next_y = x, y
                                            front_x, front_y = x + dx, y + dy
                                            #can only pick up key if on the key cell and not already carrying a key
                                            if (front_x,front_y) == (key_pos[0], key_pos[1]) and key_state == 0:
                                                next_key = 1
                                            else:
                                                next_key = key_state
                                            next_door1 = door1_state
                                            next_door2 = door2_state
                                        if action == UD:
                                            dx, dy = dir_dict[dir]
                                            next_dir = dir
                                            front_x, front_y = x + dx, y + dy
                                            next_x, next_y = x, y
                                            next_key = key_state
                                            next_door1 = door1_state
                                            next_door2 = door2_state
                                            if (front_x, front_y) == (5, 3) and key_state == 1 and door1_state == 0:
                                                next_door1 = 1
                                            elif (front_x, front_y) == (5, 7) and key_state == 1 and door2_state == 0:
                                                next_door2 = 1

                                        
                                        key_idx = Keys.index(key_pos)
                                        goal_idx = Goals.index(goal_pos)

                                        #compute state scalar 
                                        state = x + W*(y + H*(dir + 4*(key_state + 2*(door1_state + 2*(door2_state + 2*(key_idx + 3*goal_idx))))))
                                        if state not in Pf:
                                            Pf[state] = {}
                                        next_state = next_x + W*(next_y + H*(next_dir + 4*(next_key + 2*(next_door1 + 2*(next_door2 + 2*(key_idx + 3*goal_idx))))))
                
                                        #can index ouut of this using the state value 
                                        cost = 1
                                        Pf[state][action] = (next_state, cost)

    return Pf

In [107]:
def dynamic_programming_rand(Pf, gamma):
    """"
    Dynamic Programming Implementation:
    Inputs: gamma
    State space is so large, that state to state transitions only happen within the same key and goal config.
    Does not use env as input 
    We assume the envirnoment is not partially observable, and we learn the pf from the env directly.
    Returns: V, policy """
    #load env, extract dictionary to built state space 
    
    H = 10
    W = 10
    Goals = [(6, 1), (7, 3), (6, 6)]
    # Doors = [(5, 3), (5, 7)]
    Keys = [(2, 2), (2, 3), (1, 6)]

    
    #time horizon 
    T = W * H 
    
    MF = 0  # Move Forward
    TL = 1  # Turn Left
    TR = 2  # Turn Right
    PK = 3  # Pickup Key
    UD = 4  # Unlock Door

    # Pf = gen_pf() #does not include goal states
    #pf has form pf[state][action] = (next_state, cost) the transitions are deterministic 
    num_states = W*H*4*2*2*2*3*3 #total states not just walkable  
    num_actions = 5

    
    q = 0 #terminal cost
    
    #acts as V_T for the final time step 
    V = np.full(num_states, np.inf) #populate initial V with inf, since we want to minimuze cost
    #set V of goal states to 0
    #caluclate goal states 
    goal_states= []
    for x in range(W):
        for y in range(H):
            for dir in range(4):
                for key_state in [0,1]: #binary has key or doesnt TRUE or FALSE
                        for door1_state in [0, 1]:
                            for door2_state in [0, 1]:
                                for key_pos in [(2, 2), (2, 3), (1, 6)]:
                                    for goal_pos in [(6, 1), (7, 3), (6, 6)]:   
                                        key_idx = Keys.index(key_pos)
                                        goal_idx = Goals.index(goal_pos)                                     
                                        state = x + W*(y + H*(dir + 4*(key_state + 2*(door1_state + 2*(door2_state + 2*(key_idx + 3*goal_idx))))))
                                        if (x,y) == goal_pos:
                                            V[state] = q
                                            goal_states.append(state)

    goal_states = set(goal_states) #for faster lookup

    pi = np.zeros((T, num_states), dtype=int) #initialize policy array
    for t in range(T-1 , -1, -1): #loop from T-1 to 0
        q_table = np.full((num_states, num_actions), np.inf)
        for x in range(num_states):
            if x not in Pf:
                continue
            if x in goal_states:
                continue
            for u in range(num_actions):
                q_table[x, u] = Pf[x][u][1] + gamma * V[Pf[x][u][0]] #prob of next state #cost(from pf) + gamma * V(next state)
                #not all states in V have a Pf counterpart, keep it at inf
            
            V_t = np.min(q_table[x, :]) #find the min cost action for each state

            #once action row has been computed for each state, update V and pi
            V[x] = V_t #update V for the current state
                #extract policy
            pi[t, x] = np.argmin(q_table[x, :]) #find the action that minimizes cost for each state

    return V, pi


In [108]:
def test_policy_rand(pi, Pf, env, info):
    """Runs using env and info passed in from generating a random environment"""
    #s = x + W * (y + H * (dir + 4 * (key_state + 2 * door_state)))
    
    
    H = 10
    W = 10
    x, y = env.agent_pos 
    dir = env.agent_dir
    key_state = int(env.carrying is not None)
    door1 = env.grid.get(5, 3)
    door2 = env.grid.get(5, 7)
    door1_state = int(not door1.is_locked)
    door2_state = int(not door2.is_locked)

    Keys = [(2, 2), (2, 3), (1, 6)]
    Goals = [(6, 1), (7, 3), (6, 6)]

    key_idx = Keys.index(tuple(info['key_pos']))
    goal_idx = Goals.index(tuple(info['goal_pos']))

    state = x + W*(y + H*(dir + 4*(key_state + 2*(door1_state + 2*(door2_state + 2*(key_idx + 3*goal_idx))))))
    sequence = []

    goal_x, goal_y = info['goal_pos']
    for t in range(len(pi)):
        if x == goal_x and y == goal_y:
            break
        action = pi[t, state]
        sequence.append(action)
        state = Pf[state][action][0]


        x = state % W
        y = (state // W) % H
    return sequence


In [109]:
from gymnasium.envs.registration import register
from minigrid.envs.doorkey import DoorKeyEnv

class DoorKey10x10Env(DoorKeyEnv):
    def __init__(self, **kwargs):
        super().__init__(size=10, **kwargs)

register(
    id='MiniGrid-DoorKey-10x10-v0',
    entry_point='__main__:DoorKey10x10Env'
)

/Users/iuliarusu/miniconda3/envs/ece276bpr1/lib/python3.11/site-packages/gymnasium/envs/registration.py:693: UserWarning: WARN: Overriding environment MiniGrid-DoorKey-10x10-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [111]:
#all together

rand_Pf = gen_pf_rand()
V, pi = dynamic_programming_rand(rand_Pf, gamma=1)

for i in range(1, 36):
    env_path = f"./envs/random_envs/DoorKey-10x10-{i}.env"
    env, info = load_env(env_path)
    seq = test_policy_rand(pi, rand_Pf, env=env, info=info)
    env_name = f"DoorKey-10x10-{i}"
    draw_gif_from_seq(seq, env, path=f"./gif/{env_name}.gif")




GIF is written to ./gif/DoorKey-10x10-1.gif
GIF is written to ./gif/DoorKey-10x10-2.gif
GIF is written to ./gif/DoorKey-10x10-3.gif
GIF is written to ./gif/DoorKey-10x10-4.gif
GIF is written to ./gif/DoorKey-10x10-5.gif
GIF is written to ./gif/DoorKey-10x10-6.gif
GIF is written to ./gif/DoorKey-10x10-7.gif
GIF is written to ./gif/DoorKey-10x10-8.gif
GIF is written to ./gif/DoorKey-10x10-9.gif
GIF is written to ./gif/DoorKey-10x10-10.gif
GIF is written to ./gif/DoorKey-10x10-11.gif
GIF is written to ./gif/DoorKey-10x10-12.gif
GIF is written to ./gif/DoorKey-10x10-13.gif
GIF is written to ./gif/DoorKey-10x10-14.gif
GIF is written to ./gif/DoorKey-10x10-15.gif
GIF is written to ./gif/DoorKey-10x10-16.gif
GIF is written to ./gif/DoorKey-10x10-17.gif
GIF is written to ./gif/DoorKey-10x10-18.gif
GIF is written to ./gif/DoorKey-10x10-19.gif
GIF is written to ./gif/DoorKey-10x10-20.gif
GIF is written to ./gif/DoorKey-10x10-21.gif
GIF is written to ./gif/DoorKey-10x10-22.gif
GIF is written to .